# 2장. Custom Guardrails — After Agent: 출력 필터 (모델 기반 검증)

**After Agent Guardrail**은 LLM이 답변을 생성한 **직후**, 사용자에게 보여주기 전에 내용을 검증합니다.

이번 예시에서는 교육용 튜터 Agent에서 AI가 정답을 직접 알려주는 것을 **감시자 LLM이 탐지하고 차단**합니다.

**동작 흐름:**
1. 메인 Agent가 답변 생성
2. `@after_agent` 미들웨어 실행 → `safety_model`(감시자 LLM)에게 답변 평가 요청
3. 감시자가 `"LEAKED"` 판정 시 → `last_message.content`를 교육적인 힌트로 교체

> **Before vs After Guardrail**  
> Before: 비용 저렴, LLM 호출 자체를 막음 → 규칙 기반(키워드/정규식)  
> After: 추가 비용 발생, 더 미세한 문제 탐지 → 모델 기반(LLM 평가)

In [5]:
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent


# .env 파일에서 환경 변수 로드
load_dotenv()



True

In [6]:
from langchain.agents.middleware import after_agent
from langchain.chat_models import init_chat_model
from langchain.messages import AIMessage

# 평가용 판사(Judge) 모델 초기화
safety_model = init_chat_model("gpt-5-mini")

@after_agent
def answer_leakage_guardrail(state, runtime) :
    """
    AI가 답변을 생성한 '직후', 사용자에게 보여주기 전에 내용을 검사.
    만약 AI가 문제의 정답을 직접적으로 말해버렸다면, 이를 감지하고 수정.
    """

    # 1. 메시지 유효성 검사
    if not state["messages"]: return None
    last_message = state["messages"][-1]

    # 마지막 메시지가 AI의 답변이 아니면 검사할 필요 없음
    if not isinstance(last_message, AIMessage):
        return None

    # 2. 감시자 AI에게 평가 요청 (Prompt Engineering)
    # 메인 AI의 답변이 교육적으로 적절한지(정답을 바로 주지 않았는지) 평가합니다.
    auditor_prompt = f"""
    당신은 엄격한 교육 감독관입니다.
    다음 '튜터의 답변'을 확인하세요.
    답변이 학생을 지도하지 않고 문제의 정답이나 전체 풀이를 직접적으로 제공한다면 'LEAKED'라고 답하세요.
    답변이 적절한 힌트나 설명을 제공한다면 'SAFE'라고 답하세요.

    튜터의 답변: {last_message.content}
    """

    result = safety_model.invoke([{"role": "user", "content": auditor_prompt}])

    # 3. 결과에 따른 개입 (Intervention)
    if "LEAKED" in result.content:
        print(f"🚨 [가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.")
        last_message.content = "앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요? 이 문제의 핵심 개념은..."

    return None


In [7]:
agent = create_agent(
    model="gpt-5-nano",
    tools=[],
    middleware=[answer_leakage_guardrail],
)
agent.invoke({
    "messages": [{"role": "user", "content": "직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 이 문제 너무 어려워. 그냥 정답 알려줘."}]
})


🚨 [가드레일 발동] 정답 유출 감지됨! 답변을 수정합니다.


{'messages': [HumanMessage(content='직각 삼각형 두 직각변의 길이가 3과 4라면 빗변의 길이가 뭐야? 이 문제 너무 어려워. 그냥 정답 알려줘.', additional_kwargs={}, response_metadata={}, id='5fcb36b0-112e-4efc-9963-2248ef22035e'),
  AIMessage(content='앗, 제가 정답을 바로 말할 뻔했네요! 😅 정답보다는 푸는 방법을 먼저 생각해볼까요? 이 문제의 핵심 개념은...', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 492, 'prompt_tokens': 46, 'total_tokens': 538, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 448, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DM7iTgY9CK1gkPXcf484AEDwNeuy7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d147d-4b28-74d1-85c0-cfc53393f9a3-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 46, 'output_tokens': 492, 'total_tokens':